# CONFIG

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import numpy as np
import os

from maikol_utils.print_utils import print_separator, print_color
from src.config import Configuration
from src.data import (
    generate_all_features,
    precompute_feature_tensors,
    compute_features_dataset,
    load_gb_images,
    create_face_augmentor,
)
from src.model import (
    CascadeSerializer,
    build_haar_cascade_from_stages,
)
from scripts import generate_all_stages


CONFIG = Configuration(
    max_stages=15,
    target_fpr=0.00005,
)

np.random.seed(CONFIG.seed)
print(CONFIG)

# 1. GENERATE HAAR FEATURES

Each feature is defined once (no white-black / black-white duplicates since AdaBoost stumps handle the polarity).

In [ ]:
all_features = generate_all_features(
    win_w=CONFIG.crop_size,
    win_h=CONFIG.crop_size,
    edge_margin=CONFIG.feature_edge_margin,
    stride=CONFIG.feature_stride,
    include_square_features=CONFIG.include_square_features,
)
print(f"Generated {len(all_features)} Haar features")

# 2. LOAD DATA

- **Faces**: from WIDER or VPC dataset
- **Background**: from Open Images v7 (via `load_gb_images`)

In [ ]:
from maikol_utils.file_utils import list_dir_files

face_path = CONFIG.faces_vpc_path if CONFIG.use_vpc_faces else CONFIG.faces_train_path
all_faces, n_faces = list_dir_files(face_path, recursive=True)
if len(all_faces) > CONFIG.max_faces > 0:
    all_faces = np.random.choice(all_faces, size=CONFIG.max_faces, replace=False)
print(f"Faces: {len(all_faces)} images from {face_path}")

bg_dataset = load_gb_images(CONFIG)
print(f"Background: {len(bg_dataset)} images")

# 3. PRECOMPUTE FACE FEATURES

Cached as `faces.npy` — reloaded on subsequent runs unless `force_features=True`.

In [ ]:
face_augmentor = create_face_augmentor(CONFIG)

if os.path.exists(CONFIG.faces_np_path) and not CONFIG.force_features:
    print(f"Loading cached face features from {CONFIG.faces_np_path}...")
    X_train_faces = np.load(CONFIG.faces_np_path)
    precomputed = precompute_feature_tensors(all_features)
else:
    X_train_faces, precomputed = compute_features_dataset(
        all_faces, all_features,
        n_workers=CONFIG.max_cpu_cores,
        augment_fn=face_augmentor,
    )
    np.save(CONFIG.faces_np_path, X_train_faces)

print(f"X_train_faces: dtype={X_train_faces.dtype}, shape={X_train_faces.shape}")

# 4. TRAIN VIOLA-JONES CASCADE

Each stage:
1. **Hard negative mining** — run current cascade on background images; collect false positives
2. **AdaBoost training** with early stopping on FPR
3. **Test macro FPR** — stop when reaching `target_fpr`

Supports checkpointing via `resume_training=True`.

In [ ]:
stages, fpr_macro = generate_all_stages(
    CONFIG,
    X_train_faces=X_train_faces,
    bg_samples=bg_dataset,
    all_features=all_features,
    precomputed=precomputed,
)

print_color(f"Finished: {len(stages)} stages, macro FPR={fpr_macro:.6f}", color="green")

# 5. BUILD & SAVE CASCADE

In [ ]:
haar_cascade = build_haar_cascade_from_stages(
    stages_output=stages,
    all_features=all_features,
    width=CONFIG.crop_size,
    height=CONFIG.crop_size,
    cascade_type="trained_adaboost_stages",
    feature_type="HAAR",
)

print_separator("FINAL CASCADE", sep_type="LONG")
print(f"  Stages:    {len(haar_cascade.stages)}")
print(f"  Features:  {len(haar_cascade.features)}")
print(f"  Window:    {haar_cascade.width}x{haar_cascade.height}")

cascade_path = os.path.join(
    CONFIG.computed_haar_cascades,
    f"stages_vj-{fpr_macro:.4f}.xml",
)
CascadeSerializer.save(haar_cascade, cascade_path)
print(f"  Saved to:  {cascade_path}")

In [ ]:
loaded = CascadeSerializer.load(cascade_path)
print(f"Verified: {len(loaded.stages)} stages, {len(loaded.features)} features")